# Tardis eda : initialisation

- modules calls
- define exit values

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pickle import dump
from os import getcwd, mkdir, listdir, chmod

EPITECH_SUCCESS = 0
EPITECH_FAILURE = 84

FEATURES_COLUMNS = [
    "Date",
    "Month",
    "Year",
    "Service",
    "Departure station",
    "Arrival station",
]
# , "Number of scheduled trains", "Number of cancelled trains"]
TARGETS_COLUMNS = ["Average delay of all trains at arrival"]
ADDED_COLUMNS = ["Month", "Year"]

SHOW = False

# Tardis part 1, cleaning the dataset:

#### Function to drop given rows :

arguments :
- csv (panda DataFrame), the initial dataframe
- rows_to_drop (panda DataFrame), the rows that will have to be dropped

Return a copy of the csv dataframe without the rows contained into the
    rows_to_drop dataframe.

In [2]:
def drop_rows(csv: pd.DataFrame, rows_to_drop: pd.DataFrame) -> pd.DataFrame:
    return csv.drop(index=rows_to_drop.index).copy()

#### Function to drop empty and uneeded columns :

argument :
- csv (panda DataFrame), the dataframe

Drop every empty columns and uneeded :
- every columns starting with 'Unnamed:' because that means we don't know what their content corresponds to.
- every columns ending by ' comments' because we cannot use the comments in our model.
- every columns starting by 'Pct ' because what the percentages point to is amibuous.

In [3]:
def drop_empty_and_uneeded_columns(csv: pd.DataFrame) -> pd.DataFrame:
    unnamed_cols = [c for c in csv.columns if c.startswith("Unnamed:")]
    pct_cols = [c for c in csv.columns if c.startswith("Pct ")]
    comment_cols = [c for c in csv.columns if c.endswith(" comments")]
    cols_to_drop = unnamed_cols + pct_cols + comment_cols

    csv.drop(columns=cols_to_drop, inplace=True, errors="ignore")
    return csv

#### Function to set the type of columns :

argument :
- csv (panda DataFrame), the dataframe

Set every columns types and return the updated dataframe :
- Set type to numeric for :
    - Average journey time
    - Number of scheduled trains
    - Number of cancelled trains
    - Number of trains delayed at departure
    - Average delay of late trains at departure
    - Average delay of all trains at departure
    - Number of trains delayed at arrival
    - "Average delay of late trains at arrival
    - Average delay of all trains at arrival
    - Number of trains delayed > 15min
    - Average delay of trains > 15min (if competing with flights)
    - Number of trains delayed > 30min
    - Number of trains delayed > 60min
- Set type to date for :
    - Date

For the numeric columns, the rows containing negative values are removed.

In [4]:
def set_type_of_columns(csv: pd.DataFrame) -> pd.DataFrame:
    columns = (
        "Average journey time",
        "Number of scheduled trains",
        "Number of cancelled trains",
        "Number of trains delayed at departure",
        "Average delay of late trains at departure",
        "Average delay of all trains at departure",
        "Number of trains delayed at arrival",
        "Average delay of late trains at arrival",
        "Average delay of all trains at arrival",
        "Number of trains delayed > 15min",
        "Average delay of trains > 15min (if competing with flights)",
        "Number of trains delayed > 30min",
        "Number of trains delayed > 60min",
    )

    for c in columns:
        csv[c] = pd.to_numeric(csv[c], errors="coerce")
        csv = csv[csv[c] >= 0.0]
    csv["Date"] = pd.to_datetime(csv["Date"], format="%Y-%m", errors="coerce")
    return csv

#### Function to remove extra whitespaces from string values :

argument :
- csv (panda DataFrame), the dataframe

Remove extra whitespaces and return the updated dataframe.

In [5]:
def remove_extra_whitespaces(csv: pd.DataFrame) -> pd.DataFrame:
    for col in ("Departure station", "Arrival station", "Service"):
        if col in csv.columns:
            csv[col] = csv[col].str.strip()
    return csv

#### Function to fix the station name containing the words "Saint" and "Sainte"

several stations names containing the words "Saint" and "Sainte" are written with different orthographs ("Saint" or "St" / "Sainte" or "Ste")

argument :
- csv (panda DataFrame), the dataframe

return the fixed dataframe

In [6]:
def get_correct_sainte_orthograph(tmp: str) -> str:
    if not "STE" in tmp:
        return tmp
    i = tmp.index("STE")
    if (i + 2) >= len(tmp):
        return tmp
    if tmp[i + 2] not in " -":
        return tmp[: i + 2] + get_correct_name(tmp[i + 2 :])
    if tmp[i + 2] == "-":
        tmp = tmp[: i + 2] + " " + tmp[i + 3 :]
    return tmp[:i] + "SAINTE" + get_correct_name(tmp[i + 2 :])


def get_correct_saint_orthograph(tmp: str) -> str:
    if not "ST" in tmp:
        return tmp
    i = tmp.index("ST")
    if (i + 2) >= len(tmp):
        return tmp
    if tmp[i + 2] not in " -":
        return tmp[: i + 2] + get_correct_name(tmp[i + 2 :])
    if tmp[i + 2] == "-":
        tmp = tmp[: i + 2] + " " + tmp[i + 3 :]
    return tmp[:i] + "SAINT" + get_correct_name(tmp[i + 2 :])


def get_correct_name(init: str) -> str:
    if type(init) != str:
        return init
    init = init.upper()
    init = get_correct_saint_orthograph(get_correct_sainte_orthograph(init))
    return init


def fix_stations_names(csv: pd.DataFrame) -> pd.DataFrame:
    dep_col = list(csv.columns).index("Departure station")
    arr_col = list(csv.columns).index("Arrival station")
    for i in range(csv.shape[0]):
        for col in (dep_col, arr_col):
            csv.iloc[i, col] = get_correct_name(csv.iloc[i, col])
    return csv

#### Function to remove to rows containing missing fields :

argument :
- csv (panda DataFrame), the dataframe

Return a copy of the csv dataframe without the rows containing invalid values.

In [7]:
def remove_rows_missing_fields(csv: pd.DataFrame) -> pd.DataFrame:
    columns = [
        c for c in (FEATURES_COLUMNS + TARGETS_COLUMNS) if c not in ADDED_COLUMNS
    ]
    for col in columns:
        csv = drop_rows(csv, csv[csv[col].isna()])
        csv = drop_rows(csv, csv[csv[col].isnull()])
        csv = drop_rows(csv, csv.query(f"`{col}` == '' or `{col}` == '0'"))
        if col not in ("Average delay of all trains at arrival", "Date"):
            csv = drop_rows(csv, csv.query(f"`{col}` == 0"))
    return csv

#### Function to remove inconsistent train numbers

argument :
- csv (panda DataFrame), the dataframe

Return a copy of the csv dataframe without the rows whose values are inconsistent (for instance if the number of trains delayed is greater than the total number of trains).

In [8]:
def remove_inconsistent_train_numbers(csv: pd.DataFrame) -> pd.DataFrame:
    nbr_trains_delayed_tot = "Number of trains delayed"
    nbr_trains_delayed_dep = "Number of trains delayed at departure"
    nbr_trains_delayed_arr = "Number of trains delayed at arrival"
    nbr_scheduled_trains = "Number of scheduled trains"
    rows_to_drop = (
        f"`{nbr_trains_delayed_dep}` > `{nbr_scheduled_trains}`",
        f"`{nbr_trains_delayed_arr}` > `{nbr_scheduled_trains}`",
        f"`{nbr_trains_delayed_tot} > 15min` > `{nbr_scheduled_trains}`",
        f"`{nbr_trains_delayed_tot} > 30min` > `{nbr_scheduled_trains}`",
        f"`{nbr_trains_delayed_tot} > 60min` > `{nbr_scheduled_trains}`",
    )
    for r in rows_to_drop:
        csv = drop_rows(csv, csv.query(r))
    return csv

#### Part 1 main function to clean the dataset:

- Cleaning steps:
  1. First, we drop empty and uneeded columns
  2. Make Average journey time float : some of them have "min" at the end, making them strings.
  3. Convert data types.
  4. Drop rows with a negative value.
  5. Remove Whitespaces from columns that have extra.
  6. Drop important rows with missing fields.
  7. Drop rows with a impossible journey time
  8. Drop rows where any delayed-train count exceeds scheduled trains

In [9]:
def part_one(csv: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
        argument : csv (panda DataFrame), the dataframe
    x
        Tardis project Step 1: Data Cleaning
        Cleans the dataset and return_it, more detail on the cleaning at the top of
            the jupyter notebook.
    """
    csv = drop_empty_and_uneeded_columns(csv)
    csv["Average journey time"] = (
        csv["Average journey time"]
        .str.replace(r"\s*min\s*$", "", regex=True)
        .str.strip()
    )
    csv = set_type_of_columns(csv)
    csv = remove_extra_whitespaces(csv)
    csv = remove_rows_missing_fields(csv)
    csv = drop_rows(csv, csv.query("`Average journey time` <= 0"))
    csv = remove_inconsistent_train_numbers(csv)
    csv = drop_rows(csv, csv.query("`Average delay of all trains at arrival` < 0"))
    csv["Year"] = csv["Date"].dt.year
    csv["Month"] = csv["Date"].dt.month
    csv["Day"] = csv["Date"].dt.day
    csv = csv.drop_duplicates()
    csv = fix_stations_names(csv)
    csv.to_csv("cleaned_dataset.csv", index=False)
    file = open("cleaned_dataset.pkl", mode="w+b")
    dump(csv, file)
    file.close()
    csv_cleaned = csv[FEATURES_COLUMNS + TARGETS_COLUMNS]
    if SHOW:
        csv.info()
        csv_cleaned.info()
    csv_cleaned.to_csv("filtered_dataset.csv", index=False)
    file = open("filtered_dataset.pkl", mode="w+b")
    dump(csv_cleaned, file)
    file.close()
    return csv, csv_cleaned

# Tardis part 2, data organizing and analyzing:

#### Making average arrival delay plot :

argument :
- csv (panda DataFrame), the dataframe

Creates and saves a plot that shows which arrival delay is the highest. \
Doesn't return anything.

In [10]:
def average_arrival_delay(csv: pd.DataFrame) -> None:
    sns.histplot(csv["Average delay of all trains at arrival"], bins=40)
    plt.title("Delay lengths?")
    plt.xlabel("Average arrival delay (minutes)")
    plt.ylabel("Number of records")
    if SHOW:
        plt.show()
    plt.savefig("Plots/Average_arrival_delay.png")
    plt.close()

#### Making average yearly delay plot

argument :
- csv (panda DataFrame), the dataframe

Creates and saves a plot that shows which an average delay per year. \
Doesn't return anything.

In [11]:
def average_yearly_delay(csv: pd.DataFrame) -> None:
    yearly_delay = csv.groupby("Year")["Average delay of all trains at arrival"].mean()
    plt.plot(yearly_delay.index, yearly_delay.values, marker="o")
    plt.title("Average arrival delay per year")
    plt.xlabel("Year")
    plt.ylabel("Average delay (minutes)")
    if SHOW:
        plt.show()
    plt.savefig("Plots/Average_yearly_delay.png")
    plt.close()

#### Making a latest delays csv + Making a top 10 departure and arrival stations delay graph :

- ##### top 10 departure delays :
argument :
- csv (panda DataFrame), the dataframe

Makes a csv for the latest delays. \
Makes a graph for the top 10 departure stations and their delay. \
Doesn't return anything.


- ##### top 10 arrival delays :
argument :
- csv (panda DataFrame), the dataframe

Makes a csv for the latest delays. \
Makes a graph for the top 10 arrival stations and their delay. \
Doesn't return anything.

In [12]:
def top_10_departure_stations_with_the_most_delays(csv: pd.DataFrame) -> None:
    station_delay = csv.groupby("Departure station")[
        "Average delay of all trains at arrival"
    ].mean()
    latest = station_delay.nlargest(10).sort_values()
    latest.plot(kind="barh")
    plt.tight_layout(pad=2.0)
    plt.title("Top 10 Departure Stations and delay")
    plt.xlabel("Average arrival delay (minutes)")
    if SHOW:
        plt.show()
    plt.savefig("Plots/Top_10_worst_depature_stations.png")
    plt.close()


def top_10_arrival_stations_with_the_most_delays(csv: pd.DataFrame) -> None:
    station_delay = csv.groupby("Arrival station")[
        "Average delay of all trains at departure"
    ].mean()
    latest = station_delay.nlargest(10).sort_values()
    latest.plot(kind="barh")
    plt.tight_layout(pad=2.0)
    plt.title("Top 10 Arrival Stations and delay")
    plt.xlabel("Average departure delay (minutes)")
    if SHOW:
        plt.show()
    plt.savefig("Plots/Top_10_worst_arrival_stations.png")
    plt.close()

#### Adding cancellation rate column

argument :
- csv (panda DataFrame), the dataframe

Adds a column for cancellation rate. \
Doesn't return anything.

In [13]:
def yearly_cancellation_rate(csv: pd.DataFrame) -> None:
    csv["cancellation_rate"] = (
        csv["Number of cancelled trains"] / csv["Number of scheduled trains"] * 100
    )
    yearly_cancel = csv.groupby("Year")["cancellation_rate"].mean()
    plt.bar(yearly_cancel.index, yearly_cancel.values)
    plt.title("Cancellation Rate per Year (%)")
    plt.xlabel("Year")
    plt.ylabel("Cancellation rate (%)")
    if SHOW:
        plt.show()
    plt.savefig("Plots/Cancellation_rate.png")
    plt.close()

#### Making a correlation heatmap :

argument :
- csv (panda DataFrame), the dataframe

Makes a heatmap for the correlation between delays. \
Doesn't retrun anything.

In [14]:
def make_correlation(csv: pd.DataFrame) -> None:
    cols_to_check = [
        "Average delay of all trains at arrival",
        "Average delay of all trains at departure",
        "Number of cancelled trains",
        "Number of trains delayed > 15min",
        "Number of trains delayed > 30min",
        "Number of trains delayed > 60min",
    ]
    plt.figure(figsize=(8, 6))
    sns.heatmap(csv[cols_to_check].corr(), annot=True, fmt=".2f", cmap="icefire")
    plt.title("Correlation between delays")
    plt.tight_layout()
    if SHOW:
        plt.show()
    plt.savefig("Plots/Correlation.png")
    plt.close()

#### Create the 'Plots' directory (if it doesn't already exists) to save plots

arguments :
- name (str), the folder name

create the folder if it doesn't already exists\
in every case, set the folder's permissions to 777\
does not return anything

In [15]:
def create_plot_dir(name: str) -> None:
    cwd = getcwd()
    path = f"{cwd}/{name}"
    if name not in listdir(cwd):
        mkdir(path)
    chmod(path, 0o777)

#### Part 2 main function to organize and analyze data:

- Data organisation:
  1. We plot the average arrival delay, to see what is the most common delay
  2. Then we plot the average yearly delay to see which years were the highest so we know when the SNCF did right things.
  3. Then we see which departure stations are the worst to know where to focus our efforts.
  4. Then we look at the yearly rate of cancelled trains to know which years we did wrong to not repeat the mistakes of the past.
  5. Finally, we make a heatmap for correlation.
- Data analysis:
  1. We can see that the delay sits at an average of about 10 to 15 minutes which is not too bad but still a cause for concern.
  2. We can see that the average delay was the lowest during 2021 which is when the coronavirus was at its strongest so there were a lot less people travelling, this can tell us that we need to work at better managing the masses since the delay is lowest when there are less people.
  3. The worst departure stations are either in the south of france or outside france, we cannot do anything about the stations outside of france but we now we need to focus on the stations in big cities on the south of France like Montpellier.
  4. We can see that the number of trains cancelled have been going down so there have been some improvements there.
  5. This tells us very similar things to the first plot.

In [16]:
def part_two(csv: pd.DataFrame) -> pd.DataFrame:
    """
    argument : csv (panda DataFrame), the dataframe

    Tardis project Step 2: Data Visualization & Analysis
    Creates and saves plots for data visualization & Analysis
    """
    create_plot_dir("Plots")
    average_arrival_delay(csv)
    average_yearly_delay(csv)
    top_10_departure_stations_with_the_most_delays(csv)
    top_10_arrival_stations_with_the_most_delays(csv)
    yearly_cancellation_rate(csv)
    make_correlation(csv)

# Tardis eda main program:

In [17]:
def tardis_eda(
    filename: str, separator: str = ","
) -> tuple[pd.DataFrame, pd.DataFrame]:
    try:
        csv = pd.read_csv(filename, sep=separator)
    except:
        return None
    return part_one(csv)


def main_eda() -> int:
    """
    no argument

    main part of the program
    """
    csv = tardis_eda("dataset.csv", ";")
    if csv is None:
        return EPITECH_FAILURE
    csv = csv[0]
    part_two(csv)
    return EPITECH_SUCCESS


if __name__ == "__main__":
    assert get_correct_name("Marseille St Charles") == "Marseille Saint Charles".upper()
    assert get_correct_name("St-Pierre des Corps") == "Saint Pierre des Corps".upper()
    assert (
        get_correct_name("St Michel - St Germain")
        == "Saint Michel - Saint Germain".upper()
    )
    main_eda()